# Optimizers and batch sizes

> **Overview.** Does Figure 1's gap between block-sparse and dense networks (24 hidden units each, 4-task multi-MNIST) hold with Adam, RMSProp and minibatch SGD?

These plots use the step-size sweeps (`experiments/sweeps/05_optimizers_and_batch_size/sweep/`, 5 seeds, 500K samples), each configuration at its best step size, so they show seed means without confidence bands. The 30-seed runs replace them later. Step sizes are chosen on last-10% test accuracy for the stationary problem and on last-10% online accuracy for the non-stationary one.

In [ ]:
# Refresh the export when needed.
# from multi_mnist.analysis import download_project
# download_project('paper-weight-pruning-rebuttal', output_dir='data')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from multi_mnist.analysis import (
    asymptotic_metric, filter_sweep, filter_to_best, get_color_palette, label_curves_inline, load_export,
    set_style,
)

PALETTE = get_color_palette(classes=['Block-Sparse', 'Dense'])
NETWORKS = [('Block-sparse', 'block_sparse', PALETTE['Block-Sparse']), ('Dense', 'dense', PALETTE['Dense'])]
CELL = ['optimizer.name', 'model.init_strategy', 'train.batch_size']
SELECT = {'stationary': 'test_accuracy', 'nonstationary': 'accuracy'}   # step-size selection metric

params, metrics = load_export('paper-weight-pruning-rebuttal', data_dir='data')
params['optimizer.learning_rate'] = params['optimizer.learning_rate'].astype(float)


def best_runs(experiment, problem):
    """One trial per (optimizer, network, batch size) at its best step size, with last-10% online accuracy."""
    cfg, run = filter_sweep(params, metrics, f'07_{experiment}_{problem}_lr',
                            cell_cols=CELL + ['optimizer.learning_rate'])
    cfg, run, _ = filter_to_best(cfg, run, CELL, SELECT[problem], tail_frac=0.1)
    online = asymptotic_metric(cfg, run, 'accuracy', CELL, tail_frac=0.1)
    return cfg.merge(online, on=CELL + ['optimizer.learning_rate']), run


def curve(run, row):
    """Seed-mean online accuracy of one trial against samples seen (rows are in logging order)."""
    acc = run.loc[run['run_id'] == row['run_id'], 'accuracy'].to_numpy()
    return (np.arange(len(acc)) + 1) * row['train.log_freq'] * row['train.batch_size'], acc


BEST = {(e, p): best_runs(e, p) for e in ('optimizers', 'batch_size') for p in SELECT}
for (e, p), (cfg, _) in BEST.items():
    print(f'{e}, {p}:')
    for _, r in cfg.sort_values(CELL).iterrows():
        print(f"  {r['optimizer.name']:8s} {r['model.init_strategy']:13s} batch {r['train.batch_size']:2.0f}  "
              f"step size 2^{np.log2(r['optimizer.learning_rate']):.0f}  last-10% online accuracy {r['_metric']:.3f}")

## Optimizers

Figure 1's two panels for each optimizer, at batch size 1: online accuracy in 1000-step means over the whole stationary run (left) and non-stationary run (right). Dotted lines on the right are each network's stationary accuracy (last 10% of its stationary run). Figure 1's right panel zooms into the last 8K steps of runs logged every 50 steps; the sweep runs log every 1000, so the whole run is shown instead.

In [ ]:
set_style('default')
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
for row, (opt, opt_label) in zip(axes, [('adam', 'Adam'), ('rmsprop', 'RMSProp')]):
    for ax, problem, title in zip(row, SELECT, ['Stationary', 'Non-Stationary']):
        cfg, run = BEST[('optimizers', problem)]
        stat_cfg, _ = BEST[('optimizers', 'stationary')]
        for label, strategy, color in NETWORKS:
            r = cfg[(cfg['optimizer.name'] == opt) & (cfg['model.init_strategy'] == strategy)].iloc[0]
            ax.plot(*curve(run, r), color=color,
                    label=f"{label}, $2^{{{np.log2(r['optimizer.learning_rate']):.0f}}}$")
            if problem == 'nonstationary':
                ceiling = stat_cfg[(stat_cfg['optimizer.name'] == opt)
                                   & (stat_cfg['model.init_strategy'] == strategy)]['_metric'].item()
                ax.axhline(ceiling, linestyle=':', color=color, linewidth=2)
        ax.set_title(f'{opt_label} | {title}')
        ax.set_xlim(0, 500_000); ax.set_ylim(0, 1); ax.set_yticks(np.arange(0, 1.01, 0.2))
        ax.grid(True, alpha=0.4); ax.set_axisbelow(True)
        ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0), useMathText=True)
        ax.legend(loc='lower right')
for ax in axes[1]:
    ax.set_xlabel('Step')
for ax in axes[:, 0]:
    ax.set_ylabel('Accuracy')
fig.tight_layout()
plt.show()

## Batch sizes

SGD with batch sizes 1 to 32, each network at its best step size per batch size. Each point is the last-10% online accuracy (the last 50K samples), averaged over 5 seeds. The problem is fixed in samples, so a batch of 32 makes 25 updates per permutation.

In [ ]:
BATCHES = [1, 2, 4, 8, 16, 32]
PANELS = {  # problem: (title, y ticks, inline label (x, side) per network)
    'stationary': ('Stationary', np.arange(0.87, 0.911, 0.01),
                   {'block_sparse': (1.1, 'below'), 'dense': (1.1, 'below')}),
    'nonstationary': ('Batching Under Non-Stationarity', np.arange(0.70, 0.851, 0.05),
                      {'block_sparse': (1.1, 'below'), 'dense': (1.1, 'below')}),
}
set_style('2-col')
for problem, (title, yticks, label_at) in PANELS.items():
    cfg, _ = BEST[('batch_size', problem)]
    fig, ax = plt.subplots()
    specs = []
    for label, strategy, color in NETWORKS:
        sub = cfg[cfg['model.init_strategy'] == strategy].sort_values('train.batch_size')
        batch, acc = sub['train.batch_size'].to_numpy(), sub['_metric'].to_numpy()
        ax.plot(batch, acc, '-o', color=color)
        x = np.geomspace(batch[0], batch[-1], 400)   # the drawn line, for label clearance
        specs.append((label, *label_at[strategy], color, (x, np.interp(np.log2(x), np.log2(batch), acc))))
    ax.set_xscale('log', base=2); ax.set_xticks(BATCHES); ax.set_xticklabels(BATCHES); ax.minorticks_off()
    ax.set_xlim(0.85, 37); ax.set_ylim(yticks[0], yticks[-1]); ax.set_yticks(yticks)
    ax.set_title(title); ax.set_xlabel('Batch Size'); ax.set_ylabel('Accuracy')
    ax.grid(True); ax.set_axisbelow(True)
    fig.tight_layout()
    label_curves_inline(ax, specs)
    for fmt in ('pdf', 'png'):
        fig.savefig(f'../figures/generated/{fmt}/rev_appendix_batch_size_{problem}.{fmt}',
                    bbox_inches='tight', dpi=400)
    plt.show()

## Load the 30-seed non-stationary optimizer results

Use one complete run per cell. Repeated trials use the same seeds.


In [ ]:
from multi_mnist.analysis import COLOR_PALETTE, fetch_per_seed_arrays, plot_ci_curve

SWEEPS = {'sgd': '06_nonstationary_log50_last_step',      # Figure 1's runs
          'adam': '07_optimizers_nonstationary_log50', 'rmsprop': '07_optimizers_nonstationary_log50'}


def ns_accs(opt, strategy):
    """(50-step period, seed) online accuracy of one 30-seed non-stationary run."""
    runs = params[(params['sweep_name'] == SWEEPS[opt]) & (params['model.init_strategy'] == strategy)]
    if opt != 'sgd':
        runs = runs[runs['optimizer.name'] == opt]
    run_id = runs['run_id'].iloc[0]   # repeated trials of a cell rerun the same seeds identically
    return fetch_per_seed_arrays([run_id], cache_dir='data/per_seed_cache')[run_id]['accs']




## SGD, Adam and RMSProp, non-stationary

The same 30-seed runs with SGD added (Figure 1's runs, step sizes 2⁻⁸ and 2⁻⁹), one panel per network, in per-permutation (800-step) means with 95% bands.

In [ ]:
OPTIMIZERS = [('sgd', 'SGD', COLOR_PALETTE[2]), ('adam', 'Adam', COLOR_PALETTE[3]),
              ('rmsprop', 'RMSProp', COLOR_PALETTE[4])]
PANELS = {  # network: (title, inline label (x, side) per optimizer)
    'dense': ('Dense', {'sgd': (250_000, 'above'), 'adam': (100_000, 'below'), 'rmsprop': (300_000, 'below')}),
    'block_sparse': ('Block-Sparse', {'sgd': (60_000, 'below'), 'adam': (300_000, 'below'),
                                      'rmsprop': (300_000, 'below')}),
}
set_style('2-col')
for strategy, (title, label_at) in PANELS.items():
    fig, ax = plt.subplots()
    specs = []
    for opt, opt_label, color in OPTIMIZERS:
        accs = ns_accs(opt, strategy)
        per_perm = accs.reshape(-1, 16, accs.shape[1]).mean(axis=1)   # 16 x 50 steps = one permutation
        steps = (np.arange(len(per_perm)) + 1) * 800
        print(f'{title} {opt_label}: last-10% accuracy {per_perm[-len(per_perm) // 10:].mean():.3f}')
        plot_ci_curve(steps, per_perm, color=color, ax=ax)
        specs.append((opt_label, *label_at[opt], color, (steps, per_perm.mean(axis=1))))
    ax.set_title(title); ax.set_xlabel('Step'); ax.set_ylabel('Accuracy')
    ax.set_xlim(0, 500_000); ax.set_xticks(range(0, 500_001, 100_000)); ax.set_ylim(0.3, 0.9); ax.set_yticks(np.arange(0.3, 0.91, 0.1))
    ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0), useMathText=True)
    ax.grid(True); ax.set_axisbelow(True)
    fig.tight_layout()
    label_curves_inline(ax, specs)
    for fmt in ('pdf', 'png'):
        fig.savefig(f'../figures/generated/{fmt}/rev_appendix_optimizers_{strategy}.{fmt}',
                    bbox_inches='tight', dpi=400)
    plt.show()